# Imports and configs

In [ ]:
import os
import hashlib
from collections import Counter, defaultdict
from dataclasses import dataclass
from collections import Counter
from concurrent.futures import ThreadPoolExecutor, as_completed
from IPython.display import display

import seaborn as sns
import matplotlib.pyplot as plt
import yaml
import numpy as np
from pathlib import Path
from PIL import Image, ImageDraw
from tqdm.auto import tqdm

In [ ]:
DATA_CONFIGURATION = yaml.safe_load(
    Path("/workspaces/WebIdentification/cv_webidentification.yaml").read_text()
)
ROOT_DIR = Path("/workspaces/WebIdentification")
DATA_DIR = ROOT_DIR / DATA_CONFIGURATION["path"]
SPLITS = ["train", "val", "test"]

# Check for BBOX variation
We want to check for the following variations in the bounding boxes.
- Place in image
- Size in image
- Class distribution
- Average bbox per image
- Corruputed images

### The following sections are checks for corrupted images in a variety of ways. This can be duplicate labels, or non-normalized images. We can add more checks here as we find more ways that the data is corrupted.

In [ ]:
@dataclass
class BBox:
    class_id: int
    x_center: float
    y_center: float
    width: float
    height: float


def hash_label_line(line: str) -> str:
    return hashlib.blake2b(line.encode("utf-8"), digest_size=16).hexdigest()


num_labels = {}
label_hash = {}
bboxs = {}
line_hash_to_images = {}

for split in SPLITS:
    label_paths = list((DATA_DIR / split).glob("*.txt"))
    num_labels[split] = []
    label_hash[split] = set()
    bboxs[split] = []
    line_hash_to_images[split] = defaultdict(set)

    duplicate_lines_within_image = 0

    for label_path in label_paths:
        with open(label_path) as f:
            lines = [line.strip() for line in f if line.strip()]

        file_line_hashes = [hash_label_line(line) for line in lines]
        hashed_lines = hash(tuple(file_line_hashes))
        label_hash[split].add(hashed_lines)

        num_labels[split].append(len(lines))

        local_counts = Counter(file_line_hashes)
        if any(count > 1 for count in local_counts.values()):
            duplicate_lines_within_image += 1
            print(f"Duplicate label line(s) inside file: {label_path}")

        for line_hash in file_line_hashes:
            line_hash_to_images[split][line_hash].add(label_path.name)

        for line in lines:
            class_id, x_center, y_center, width, height = line.split()
            bboxs[split].append(
                BBox(
                    int(class_id),
                    float(x_center),
                    float(y_center),
                    float(width),
                    float(height),
                )
            )

        if len(lines) == 0:
            print(f"Corrupted image found: {label_path.with_suffix('.png')}")

        if len(hashed_lines) != len(file_line_hashes):
            print(f"Hash collision detected in file: {label_path}")

    average_bbox_per_image = (
        sum(num_labels[split]) / len(label_paths) if label_paths else 0.0
    )
    print(f"Average bbox per image for {split}: {average_bbox_per_image:.3f}")

    duplicate_label_files = len(label_paths) - len(label_hash[split])

    line_hashes_reused_across_images = {
        line_hash: image_names
        for line_hash, image_names in line_hash_to_images[split].items()
        if len(image_names) > 1
    }
    print(
        f"Files with duplicate lines inside image for {split}: {duplicate_lines_within_image}"
    )
    print(
        f"Line hashes reused across different images for {split}: {len(line_hashes_reused_across_images)}"
    )

### Show the distribution per split, 
note only buttons for now, but still gives an idea of numbers.

In [ ]:
for split in SPLITS:
    plt.figure(figsize=(10, 6))
    sns.histplot(num_labels[split], bins=20, kde=True)
    plt.title(f"Distribution of number of labels per image for {split} split")
    plt.xlabel("Number of labels")
    plt.ylabel("Frequency")
    plt.show()

### Show the distribution of where the bboxes are located in the image. 

In [ ]:
for split in SPLITS:
    x_centers = [bbox.x_center for bbox in bboxs[split]]
    y_centers = [bbox.y_center for bbox in bboxs[split]]

    plt.figure(figsize=(10, 6))
    sns.histplot(x=x_centers, y=y_centers, bins=40, cbar=True, cmap="viridis")
    plt.title(f"Heatmap of bbox centers for {split} split")
    plt.xlabel("X Center")
    plt.ylabel("Y Center")
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.show()

### Show the distribution of the bbox sizes in the image.

Note, shows them as if they were in the middle of the image, so we can see how big they are, but not where they are.

In [ ]:
IMG_W = 1920
IMG_H = 1080

for split in SPLITS:
    canvas = Image.new("RGB", (IMG_W, IMG_H), (18, 18, 24))
    draw = ImageDraw.Draw(canvas, "RGBA")

    for bbox in bboxs[split]:
        x_center = 0.5
        y_center = 0.5

        left = int((x_center - bbox.width / 2) * IMG_W)
        right = int((x_center + bbox.width / 2) * IMG_W)
        top = int((y_center - bbox.height / 2) * IMG_H)
        bottom = int((y_center + bbox.height / 2) * IMG_H)

        left = max(0, min(IMG_W, left))
        right = max(0, min(IMG_W, right))
        top = max(0, min(IMG_H, top))
        bottom = max(0, min(IMG_H, bottom))

        if right <= left or bottom <= top:
            continue

        draw.rectangle(
            [(left, top), (right - 1, bottom - 1)],
            outline=(255, 120, 40, 70),
            width=1,
        )

    center_x = IMG_W // 2
    center_y = IMG_H // 2
    draw.line([(center_x, 0), (center_x, IMG_H - 1)], fill=(80, 180, 255, 120), width=2)
    draw.line([(0, center_y), (IMG_W - 1, center_y)], fill=(80, 180, 255, 120), width=2)

    print(f"Centered bbox rectangles for {split} ({IMG_W}x{IMG_H})")
    display(canvas)

### Show "corrupted" data

In [ ]:
zero_zero_counts = {}
total_zero_zero = 0

for split in SPLITS:
    count = sum(1 for bbox in bboxs[split] if bbox.width == 0.0 and bbox.height == 0.0)
    zero_zero_counts[split] = count
    total_zero_zero += count
    print(f"0x0 boxes in {split}: {count}")

print(f"Total 0x0 boxes across all splits: {total_zero_zero}")

In [ ]:
zero_zero_counts = {}
total_zero_zero = 0

for split in SPLITS:
    count = sum(
        1 for bbox in bboxs[split] if bbox.x_center < 0.1 and bbox.y_center < 0.1
    )
    zero_zero_counts[split] = count
    total_zero_zero += count
    print(f"0x0 boxes in {split}: {count}")

print(f"Total 0x0 boxes across all splits: {total_zero_zero}")

### Show the distribution of colours in the bbox slices. 

In [ ]:
# Fixed 16-color RGB palette.
# https://en.wikipedia.org/wiki/Web_colors#Basic_colors
PALETTE = [
    ("White", (255, 255, 255)),
    ("Silver", (192, 192, 192)),
    ("Gray", (128, 128, 128)),
    ("Black", (0, 0, 0)),
    ("Red", (255, 0, 0)),
    ("Maroon", (128, 0, 0)),
    ("Yellow", (255, 255, 0)),
    ("Olive", (128, 128, 0)),
    ("Lime", (0, 255, 0)),
    ("Green", (0, 128, 0)),
    ("Aqua", (0, 255, 255)),
    ("Teal", (0, 128, 128)),
    ("Blue", (0, 0, 255)),
    ("Navy", (0, 0, 128)),
    ("Fuchsia", (255, 0, 255)),
    ("Purple", (128, 0, 128)),
]
PALETTE_RGB = np.array([rgb for _, rgb in PALETTE], dtype=np.int32)
PALETTE_MCOLORS = [tuple(v / 255 for v in rgb) for _, rgb in PALETTE]


def process_label_file(label_path):
    local_bins = Counter()
    image_path = label_path.with_suffix(".png")
    if not image_path.exists():
        return local_bins

    image = Image.open(image_path).convert("RGB")
    image_arr = np.array(image, dtype=np.uint8)

    with open(label_path) as f:
        lines = [line.strip() for line in f if line.strip()]

    for line in lines:
        class_id, x_center, y_center, width, height = line.split()
        x_center = float(x_center)
        y_center = float(y_center)
        width = float(width)
        height = float(height)

        left = int((x_center - width / 2) * image.width)
        right = int((x_center + width / 2) * image.width)
        top = int((y_center - height / 2) * image.height)
        bottom = int((y_center + height / 2) * image.height)

        left = max(0, min(image.width, left))
        right = max(0, min(image.width, right))
        top = max(0, min(image.height, top))
        bottom = max(0, min(image.height, bottom))

        if right <= left or bottom <= top:
            continue

        region = image_arr[top:bottom, left:right]
        pixels = region.reshape(-1, 3).astype(np.int32)

        diff = pixels[:, None, :] - PALETTE_RGB[None, :, :]
        dist2 = np.sum(diff * diff, axis=2)
        nearest_palette_idx = np.argmin(dist2, axis=1)

        palette_counts = np.bincount(nearest_palette_idx, minlength=len(PALETTE))
        dominant_idx = int(np.argmax(palette_counts))
        dominant_name, _ = PALETTE[dominant_idx]
        local_bins[dominant_name] += 1

    return local_bins


for split in SPLITS:
    color_bins = Counter({name: 0 for name, _ in PALETTE})
    label_paths = list((DATA_DIR / split).glob("*.txt"))

    if not label_paths:
        print(f"16-color dominant distribution for {split}: no label files found")
        continue

    max_workers = os.cpu_count() or 1

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [
            executor.submit(process_label_file, label_path)
            for label_path in label_paths
        ]
        for future in tqdm(
            as_completed(futures),
            total=len(futures),
            desc=f"{split} images",
            leave=False,
            unit="image",
        ):
            color_bins.update(future.result())

    labels = [name for name, _ in PALETTE]
    sizes = [color_bins[name] for name in labels]
    total = sum(sizes)
    sizes = [size / total * 100 for size in sizes]

    wedge_colors = PALETTE_MCOLORS
    fig, ax = plt.subplots(figsize=(14, 7))
    wedges, texts, autotexts = ax.pie(
        sizes,
        labels=labels,
        colors=wedge_colors,
        autopct="%1.1f%%",
        startangle=140,
        wedgeprops={"edgecolor": "0.3", "linewidth": 0.6},
    )
    ax.set_title(f"16-color dominant distribution for {split} split")
    ax.axis("equal")
    plt.show()